# ALORA vs LoRA Race

This notebook benchmarks two Granite Switch models — one using **ALORA** (attention-level
adapter switching) and one using standard **LoRA** — on a multi-step RAG pipeline, and
produces an animated HTML replay of the race.

The benchmark uses the government-services corpus from
[IBM mt-rag-benchmark](https://github.com/IBM/mt-rag-benchmark) as its retrieval dataset.

**What runs:**
- 16 conversations × 5 turns each, against each server
- Pipeline: guardian → query rewrite → ChromaDB retrieval → answerability → clarification → generation
- 8 concurrent requests per server, top-10 retrieved documents

**What you get:**
- `race_live.html` — animated replay with per-conversation progress bars, step colors, and vLLM metrics
- `race_report.html` — static summary with latency charts and step breakdown

**Runtime note:** Each server run takes roughly the same wall time as a real race leg
(3–8 min depending on GPU). The two runs are sequential since Colab typically provides
one GPU; `race_live.html` replays them as if they raced.

---
**Prerequisites:** GPU runtime (A100 or better). Go to *Runtime → Change runtime type → A100 GPU*.

## 0 · Install and set up

In [ ]:
# Clone the repo (skip if already present)
import os
if not os.path.exists("/content/granite-switch"):
    !GIT_TERMINAL_PROMPT=0 git clone https://github.com/ibm-granite/granite-switch.git /content/granite-switch
else:
    print("Repo already cloned")

In [ ]:
# Install granite-switch with vLLM backend and benchmark dependencies.
# This takes a few minutes on a fresh Colab instance.
%pip install -q -e "/content/granite-switch[vllm]"
%pip install -q mellea chromadb rich tqdm transformers httpx

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # needed to pull ibm-granite models from the Hub

In [ ]:
import os, subprocess, torch

RACE_DIR = "/content/granite-switch/tutorials/alora_vs_lora_race"
os.chdir(RACE_DIR)
print("Working dir:", os.getcwd())

if not torch.cuda.is_available():
    raise RuntimeError("GPU required — enable a GPU runtime in Runtime > Change runtime type")

r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("GPU:", r.stdout.strip())

## 1 · Build ChromaDB index

Downloads 49k government-service passages from the IBM mt-rag-benchmark corpus and embeds
them with `granite-embedding-small-english-r2`. Runs once and persists to `govt_chroma/`.
Skip this cell if `govt_chroma/` is already present.

In [ ]:
import os
chroma_dir = os.path.join(RACE_DIR, "govt_chroma")
if os.path.exists(chroma_dir) and os.listdir(chroma_dir):
    print(f"govt_chroma already exists ({chroma_dir}) — skipping build")
else:
    print("Building ChromaDB index (downloads ~50 MB corpus, then embeds 49k passages)...")
    !python build_govt_chroma.py

## 2 · Helper: vLLM server management

In [ ]:
import subprocess, time, requests

MAX_MODEL_LEN = "32768"  # 32k — fits comfortably on an A100 (40/80 GiB)

def launch_vllm(model, port, log_file, extra_args=None):
    cmd = [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model",         model,
        "--port",          str(port),
        "--max-model-len", MAX_MODEL_LEN,
    ]
    if extra_args:
        cmd += extra_args
    proc = subprocess.Popen(cmd, stdout=open(log_file, "w"), stderr=subprocess.STDOUT)
    print(f"Launched {model} on :{port}  (pid {proc.pid}, log → {log_file})")
    return proc


def wait_for_server(port, timeout=300):
    """Poll /health until vLLM is ready."""
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=2).status_code == 200:
                print(f"\n  ready in {int(time.time()-t0)}s")
                return True
        except Exception:
            pass
        elapsed = int(time.time() - t0)
        print(f"  waiting for :{port} ... {elapsed}s", end="\r")
        time.sleep(5)
    print(f"\n  timed out after {timeout}s — check the log file")
    return False


def tail_log(log_file, n=20):
    r = subprocess.run(["tail", f"-{n}", log_file], capture_output=True, text=True)
    print(r.stdout)

## 3 · ALORA server

Start `granite-switch-4.1-3b-preview` on port 8111 and run the benchmark against it.

In [ ]:
# Kill any stale vLLM processes from a previous run — they hold GPU memory even
# after a kernel restart and will prevent a new server from launching.
import subprocess, time, signal, os

r = subprocess.run(["pgrep", "-f", "vllm.entrypoints"], capture_output=True, text=True)
pids = [p for p in r.stdout.strip().split("\n") if p]
if pids:
    print(f"Killing stale vLLM processes: {pids}")
    for pid in pids:
        try:
            os.kill(int(pid), signal.SIGTERM)
        except ProcessLookupError:
            pass
    time.sleep(5)  # wait for processes to exit and GPU memory to be freed
else:
    print("No stale vLLM processes found.")

r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.used,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("GPU state:", r.stdout.strip())

In [ ]:
alora_proc = launch_vllm(
    model    = "ibm-granite/granite-switch-4.1-3b-preview",
    port     = 8111,
    log_file = "/content/vllm_alora.log",
)
if not wait_for_server(8111):
    tail_log("/content/vllm_alora.log")

In [ ]:
# Benchmark the ALORA server.
# --no-live disables Rich Live (which floods notebook output with redrawn frames).
# The animated replay comes from race_live.html at the end.
!python bench_pipeline_race.py --mode sequential --server "ALORA (8111)" --no-live -n 16 -c 8 -k 10

In [ ]:
alora_proc.terminate()
alora_proc.wait()
print("ALORA server stopped")

## 4 · Compose the LoRA-only model

The ALORA model above is a pre-built checkpoint from the Hub. For a fair comparison we now
compose a **LoRA-only** version from the same adapter libraries, using `--technology-filter lora`
to force every adapter to its standard LoRA variant.

This downloads the base model and adapter libraries (~6 GB on first run, cached after that)
and writes the composed checkpoint to `/content/granite-switch-lora-only`.

In [ ]:
LORA_MODEL_DIR = "/content/granite-switch-lora-only"

import os
if os.path.exists(os.path.join(LORA_MODEL_DIR, "adapter_index.json")):
    print(f"LoRA model already composed at {LORA_MODEL_DIR} — skipping")
else:
    !python -m granite_switch.composer.compose_granite_switch \
      --base-model ibm-granite/granite-4.1-3b \
      --adapters ibm-granite/granitelib-rag-r1.0 \
                 ibm-granite/granitelib-core-r1.0 \
                 ibm-granite/granitelib-guardian-r1.0 \
      --technology-filter lora \
      --output {LORA_MODEL_DIR}

## 5 · LoRA server

Start the LoRA-only composed model on port 8112 and run the same benchmark.

In [ ]:
lora_proc = launch_vllm(
    model    = LORA_MODEL_DIR,
    port     = 8112,
    log_file = "/content/vllm_lora.log",
)
if not wait_for_server(8112):
    tail_log("/content/vllm_lora.log")

In [ ]:
!python bench_pipeline_race.py --mode sequential --server "LORA (8112)" --lora-model {LORA_MODEL_DIR} --no-live -n 16 -c 8 -k 10

In [ ]:
lora_proc.terminate()
lora_proc.wait()
print("LoRA server stopped")

## 6 · Merge results into HTML

Both server runs wrote their data to `race_events.json` and `race_results.json`.
This step copies the HTML templates from `sample_run/` and embeds the data so
the replay and report work as standalone files.

In [ ]:
import json, re, shutil
from pathlib import Path

race_dir = Path(RACE_DIR)
sample   = race_dir / "sample_run"

def embed(template_name, json_name, var_name, marker):
    """Copy template from sample_run/ and embed JSON data."""
    src  = sample / template_name
    dst  = race_dir / template_name
    data = json.loads((race_dir / json_name).read_text())
    shutil.copy2(src, dst)
    html = dst.read_text()
    html = re.sub(
        rf"const {var_name} = .*?; // {marker}",
        f"const {var_name} = {json.dumps(data)}; // {marker}",
        html, flags=re.DOTALL,
    )
    dst.write_text(html)
    print(f"  {template_name}: embedded from {json_name}")

embed("race_live.html",   "race_events.json",  "RACE_EVENTS_EMBEDDED", "<<RACE_EVENTS>>")
embed("race_report.html", "race_results.json", "RACE_DATA_EMBEDDED",  "<<RACE_DATA>>")

## 7 · Results

The animated replay shows the two servers as if they had raced simultaneously — timestamps
are relative to each server's own start, so the replay is accurate for each server's
internal dynamics even though they ran back-to-back.

If the output cell is clipped, click the expand arrows on its left edge.

In [ ]:
# Animated race replay — auto-plays at 5×.
# Use the scrubber or speed buttons to explore the telemetry.
from IPython.display import HTML
HTML(open("race_live.html").read())

In [ ]:
# Static summary: step latency charts, exit distribution, per-conversation wall times.
from IPython.display import HTML
HTML(open("race_report.html").read())